# Lecture 2.4 — Dynamic Instructions: Injecting Runtime Context via a Callback

This notebook demonstrates how to upgrade an agent's `instructions` field from a static string to a **callable function** that receives live runtime data on every turn. By the end of this notebook you will have:

- passed a sync and an async callable as `instructions`
- used `RunContextWrapper` to access a custom context object inside the callable
- built three progressively realistic examples: style switching, per-user personalisation, and live date injection
- understood when dynamic instructions are the right tool and when a static string is enough

> **RunContextWrapper — first appearance.** This notebook is the first place in the course where `RunContextWrapper` appears. The same pattern will come back in Section 3 (tools), Section 5 (guardrails), and Lecture 6.4 (lifecycle hooks). It is worth pausing on it here.

## Cell 1 — Install the SDK

📌 **Notebook update notice:** in the video for this lecture, you'll hear `0.17.4` mentioned as the pinned SDK version. Since recording, a downstream dependency change (`openai>=2.45.0`, released July 9, 2026) broke `openai-agents` versions below 0.18.1 — `Runner.run()` will fail on the version stated in the video. This notebook has been updated to pin `openai-agents==0.18.3`, which fixes the issue without changing any of the code or concepts taught in the lecture. Please use the version pinned below, not the one mentioned in the recording.

The cell below installs `openai-agents` pinned to a specific version. **Pinning ensures that all the examples in this notebook run exactly as recorded**, regardless of when you open it — future SDK releases may introduce breaking changes.

If you want to use a different version:
- **Latest (no pin):** change the command to `!pip install openai-agents -q`
- **Different version:** replace `0.18.3` with the version you want, e.g. `==0.18.0`

If the package is already present in the current session at the pinned version, pip confirms this and moves on immediately — no harm done in running the cell again.

The `-q` flag suppresses most of pip's output so the cell stays readable.

In [ ]:
# Pinned for reproducibility. Updated after recording — see the
# notice above. Originally pinned to 0.17.4 as stated in the
# video; updated to 0.18.3 to fix a breaking change introduced
# by openai>=2.45.0 (July 9, 2026).
# To use the latest version instead, run: pip install openai-agents
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.0/843.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 8.6 MB/s eta 0:00:00


## Cell 2 — API Key Setup

This notebook uses **Google Colab Secrets** to load the OpenAI API key safely — no key is ever written into the notebook itself.

### How to add the secret in Colab

1. Click the **🔑 key icon** in the left sidebar (or go to **Tools → Secrets**).
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY` and paste your key as the value.
4. Toggle **Notebook access** to ON for this secret.
5. Run the cell below.

> **Running locally?** Set the environment variable in your terminal before launching Jupyter:
> ```bash
> export OPENAI_API_KEY="sk-..."
> ```
> Then replace the `userdata.get(...)` call below with `os.environ.get("OPENAI_API_KEY")`.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Imports

This cell brings in everything needed for the notebook.

| Import | Why it's here |
|---|---|
| `dataclass` | Defines lightweight, typed context objects that carry runtime state |
| `Literal` | Constrains the `style` field to a fixed set of string values — the type checker will catch typos |
| `Reasoning` | Part of `ModelSettings`; used here to disable chain-of-thought to keep responses concise in demos |
| `Agent` | The core agent class |
| `ModelSettings` | Configures model-level parameters such as `reasoning` |
| **`RunContextWrapper`** | ⭐ **First appearance in this course.** The wrapper the SDK passes into every dynamic-instructions callable (and later into tools, hooks, and guardrails). `wrapper.context` gives you your own Python object. |
| `Runner` | Executes agent runs via `await Runner.run()` |

In [ ]:
from dataclasses import dataclass
from typing import Literal

from openai.types.shared import Reasoning
from agents import Agent, ModelSettings, RunContextWrapper, Runner

## Cell 4 — Model Selection

All agents in this notebook reference a single `MODEL` variable defined here. This makes it easy to switch models in one place rather than hunting through every cell.

The default is `gpt-5.4-mini` — OpenAI's current default model for the Agents SDK (as of SDK v0.16.0+). It is fast, cost-efficient, and capable enough for all the demos in this lecture.

**To use a different model**, just change the string below and re-run this cell before running any agent cells.

> 📋 **See all available OpenAI models:** https://platform.openai.com/docs/models

In [ ]:
# Change this string to switch models across the entire notebook.
# See all available models at: https://platform.openai.com/docs/models
MODEL = "gpt-5.4-mini"

## Cell 5 — What Are Dynamic Instructions?

By default you pass `instructions` as a plain string and the agent uses it as a static system prompt for every turn. But the `instructions` field also accepts a **callable**.

When `instructions` is a callable, the SDK calls it on **every turn** — right before the model call — and uses the string it returns as the system prompt for that turn. This means the system prompt can change between turns within the same run.

### Signature rules

The callable **must** accept exactly **2 parameters**: `(run_context: RunContextWrapper[T], agent: Agent[T])`. Passing one or three parameters raises a `TypeError` at runtime — the SDK enforces this by inspecting the function signature.

Both **sync** and **async** callables are supported:

```python
# Sync callable — fine for most cases
def my_instructions(run_context: RunContextWrapper[MyContext], agent: Agent[MyContext]) -> str:
    return f"Hello, {run_context.context.username}!"

# Async callable — needed when instructions come from a DB or external service
async def my_instructions(run_context: RunContextWrapper[MyContext], agent: Agent[MyContext]) -> str:
    prompt = await fetch_from_db(run_context.context.client_id)
    return prompt
```

### What `RunContextWrapper` is — and isn't

The most important thing to know about `RunContextWrapper`:

> ⚠️ **The context object is NOT sent to the LLM.** It is a purely local Python object. The string your function *returns* becomes the system prompt; the context itself stays on your machine.

| Property | What you get |
|---|---|
| `wrapper.context` | Your app-defined Python object — whatever you passed to `Runner.run(..., context=your_object)` |
| `wrapper.usage` | Aggregated token usage so far in this run |

### Where to pass the context object

You pass your context object when you run the agent:

```python
result = await Runner.run(agent, "Your prompt", context=your_object)
```

Every agent, tool, hook, and guardrail in a run shares the same context type.

## Cell 6 — Example 1: Style-Switching Agent (Sync Callable)

This is the simplest possible dynamic-instructions pattern, adapted directly from the SDK's own example code.

### What we're building

A single agent that can respond in three different communication styles — **formal**, **casual**, or **bullet points** — driven entirely by a `StyleContext` object we pass at runtime. Same agent definition, same prompt; the only thing that changes is the context object.

### Design decisions

| Decision | Rationale |
|---|---|
| `@dataclass` for context | Lightweight, typed, no boilerplate. A Pydantic `BaseModel` works too. |
| `Literal["formal", "casual", "bullet_points"]` | The type checker will flag unknown styles at write time rather than failing silently at runtime. |
| Sync callable | No async work needed here — reading from an in-memory object is instant. |
| `reasoning=Reasoning(effort="none")` | Disables chain-of-thought to keep demo output concise. |

In [ ]:
#Define the dataclass
@dataclass
class StyleContext:
  style: Literal["formal", "casual", "bullet_points"]

def dynamic_instructions(
    run_context: RunContextWrapper[StyleContext],
    agent: Agent[StyleContext]
) -> str:

  style = run_context.context.style

  if style == "formal":
    return (
        "You are a formal business assistant."
        "Use professional language and complete sentences"
    )
  elif style == "casual":
    return (
        "You are a casual, friendly assistant."
        "Use conversational language and contractions"
    )
  else:
    return (
        "You are a concise assistant."
        "Always respond using bullet points only."
    )



#Define Dynamic instructions here


agent = Agent(
    name="Style Agent",
    instructions=dynamic_instructions,
    model=MODEL,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
    ),
)

## Cell 7 — Run the Style Agent

We loop over the three style values, construct a `StyleContext` for each, and pass it to `Runner.run()` as the `context` argument. The `context` parameter is how you inject your object into the run — the SDK then makes it available inside `run_context.context` in the callable.

Notice that only the context object changes between iterations. The agent definition, the prompt, and the model are identical each time.

In [ ]:
prompt = "Explain what an API is."

for style in ["formal", "casual", "bullet_points"]:
    #Construct the Context and pass it to the runner
    ctx = StyleContext(style=style)
    result = await Runner.run(agent, prompt, context=ctx)
    print(f"\n--- Style: {style} ---")
    print(result.final_output)


--- Style: formal ---
An API, or **Application Programming Interface**, is a set of rules that allows different software systems to communicate with each other.

In practical terms, an API defines **what requests can be made**, **how to make them**, and **what data will be returned**. It acts like a bridge between applications, enabling them to exchange information without needing to know how each other works internally.

### Simple example
If you use a weather app on your phone:
- The app sends a request to a weather service through an API.
- The weather service returns the current temperature, forecast, and other data.
- The app displays that information to you.

### Why APIs are useful
APIs help developers:
- connect different systems,
- reuse existing services,
- automate tasks,
- and build applications faster.

### Common analogy
An API is similar to a **menu in a restaurant**:
- The menu tells you what you can order.
- You place an order using those options.
- The kitchen prepar

## Cell 8 — Example 2: Per-User Personalisation (Async Callable)

The `instructions` callable can also be **async**. This is the right choice when you need to fetch the system prompt from an external source at runtime — for example, pulling a client-specific prompt template from a database, reading feature flags from a config service, or resolving the user's locale before building the string.

### What we're building

An agent that personalises its behaviour based on a `UserContext` object — addressing the user by name and adjusting response depth based on whether they are a premium subscriber.

In this example the `async` keyword is mostly illustrative: the function body doesn't actually `await` anything, because we're reading from an in-memory object. In a real application, you would `await your_db_client.get_prompt(user.client_id)` inside the function and the pattern would be identical.

### Context object fields

| Field | Type | Purpose |
|---|---|---|
| `user_id` | `str` | Unique user identifier (would be used to look up DB records) |
| `username` | `str` | Display name — addressed directly in the system prompt |
| `is_premium` | `bool` | Controls response depth and upsell messaging |

In [ ]:
import asyncio


@dataclass
class UserContext:
    user_id: str
    username: str
    is_premium: bool


async def personalised_instructions(
    run_context: RunContextWrapper[UserContext],
    agent: Agent[UserContext],
) -> str:
    user = run_context.context
    base = (
        f"You are a helpful assistant for {user.username}. "
        f"Address the user by their first name."
    )
    if user.is_premium:
        return (
            base
            + " This is a premium user — provide detailed, comprehensive answers."
        )
    return (
        base
        + " Provide concise answers. "
        + "For complex topics, suggest upgrading to premium."
    )


personalised_agent = Agent(
    name="Personalised Agent",
    instructions=personalised_instructions,
    model=MODEL,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
    ),
)

## Cell 9 — Run with Two User Profiles

We run the same agent twice — once for a free user (Alice) and once for a premium user (Bob) — using the same prompt. The agent dynamically adjusts its persona and response depth based on `is_premium`.

Key takeaway: **one agent definition handles multiple user tiers**. There is no need to clone the agent or add conditional logic outside the instructions function.

In [ ]:
free_user = UserContext(
    user_id="u001",
    username="Alice",
    is_premium=False,
)
premium_user = UserContext(
    user_id="u002",
    username="Bob",
    is_premium=True,
)

prompt = "Explain how neural networks work."

result_free = await Runner.run(
    personalised_agent,
    prompt,
    context=free_user,
)
result_premium = await Runner.run(
    personalised_agent,
    prompt,
    context=premium_user,
)

print("Free user (Alice):")
print(result_free.final_output)
print("\nPremium user (Bob):")
print(result_premium.final_output)

Free user (Alice):
Of course, Alice — here’s the short version:

A neural network is a machine learning model inspired by the brain. It’s made of layers of connected nodes called neurons.

How it works:
- Input layer: takes in data, like an image or text.
- Hidden layers: each neuron mixes the inputs using weights, adds a bias, and passes the result through an activation function.
- Output layer: produces the final prediction, like “cat” or “dog.”

Training:
- The network makes a prediction.
- It compares that prediction to the correct answer.
- A loss function measures the error.
- Backpropagation calculates how to adjust the weights.
- Gradient descent updates the weights to reduce error over time.

Why it’s useful:
- It can learn complex patterns from data.
- More layers usually mean it can model more complicated relationships.

If you want, I can also explain:
- with a simple diagram,
- with a math example,
- or how backpropagation works step by step.

For a deeper, more technical 

## Cell 10 — Example 3: Injecting Runtime Data — Current Date and Time

One of the most practical patterns in the SDK documentation is **injecting time-sensitive data** that would be stale if it were hardcoded. The current date is the canonical example — if you wrote today's date into a static `instructions` string, the agent would be wrong by tomorrow.

Other common candidates for runtime injection:

- **User locale or timezone** — e.g., format dates as `DD/MM/YYYY` for UK users
- **Feature flags** — e.g., enable a beta feature only for users in a pilot group
- **Active promotions** — e.g., mention a discount that's valid only this week
- **Server-side config** — e.g., the agent's persona is stored in a CMS and fetched at runtime

### How this cell works

`datetime.now()` is called **inside** the instructions function, so it is evaluated freshly on every run. The timezone string comes from the `DateAwareContext` object, which is passed in by the caller.

In [ ]:
from datetime import datetime


@dataclass
class DateAwareContext:
    timezone: str


def date_aware_instructions(
    run_context: RunContextWrapper[DateAwareContext],
    agent: Agent[DateAwareContext],
) -> str:
    now = datetime.now().strftime("%A, %d %B %Y %H:%M")
    tz = run_context.context.timezone
    return (
        f"You are a helpful assistant. "
        f"The current date and time is {now} ({tz}). "
        f"Use this when answering any time-sensitive questions."
    )


date_agent = Agent(
    name="Date Aware Agent",
    instructions=date_aware_instructions,
    model=MODEL,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
    ),
)

ctx = DateAwareContext(timezone="IST")
result = await Runner.run(
    date_agent,
    "What day of the week is it today?",
    context=ctx,
)
print(result.final_output)

Today is **Tuesday**.


## Cell 11 — When to Use Dynamic vs Static Instructions

Dynamic instructions add a small amount of complexity — a function definition, a context dataclass, and a `context=` argument in every `Runner.run()` call. This is the right trade-off in many situations, but not all.

| Use **static** instructions when | Use **dynamic** instructions when |
|---|---|
| The agent's role never changes | The persona or scope varies by user |
| No runtime data needed in the system prompt | You need to inject username, role, tier, date, or locale |
| Simplicity is the priority | You are fetching instructions from a database or config service |
| The agent is a singleton shared by all users | Multiple users or tenants share one agent definition |

A good heuristic: if you would need to clone the agent (`.clone()`) or create a new `Agent(...)` instance for every user, that is a signal that dynamic instructions might be the cleaner solution.

## Cell 12 — RunContextWrapper — What It Is and Isn't

Before closing, a focused summary of `RunContextWrapper` anchored to the SDK documentation.

### What it exposes

| Property | What you get |
|---|---|
| `wrapper.context` | **Your** app-defined Python object — the one you passed to `Runner.run(..., context=your_object)` |
| `wrapper.usage` | Aggregated token usage across the current run (updated after every model call) |
| `wrapper.tool_input` | Structured input when the current run is executing inside `Agent.as_tool()` (advanced) |

### The most important thing to remember

> **`RunContextWrapper` is a local Python container. It is never sent to the LLM.** The string your instructions function *returns* is what becomes the system prompt. The context object itself lives only in your Python process.

### One context type per run

Every agent, tool, hook, and guardrail within a single `Runner.run()` call must use the **same type** of context. You cannot mix `UserContext` and `DateAwareContext` in the same run — pick one dataclass that carries everything you need.

### Where you'll see this again

The `RunContextWrapper` / context injection pattern recurs throughout the SDK:

| Where | Lecture |
|---|---|
| Function tools | Section 3 |
| Input and output guardrails | Section 5 |
| Lifecycle hooks (`RunHooks`, `AgentHooks`) | Lecture 6.4 |
| Context management deep-dive | Lecture 4.6 |

The pattern is always the same: your object lives at `wrapper.context`.